# Türkçe LoRA katman deneyi: Kaggle eğitimi

Ayarlar (sağ panel): **Accelerator → GPU T4 x2**, **Internet → On**.

Uzun sürdüğü için **Save Version → Save & Run All (Commit)** ile çalıştır; tarayıcıyı kapatsan da devam eder (en fazla 12 saat).
Çıktılar `outputs/` altında notebook çıktısı olarak saklanır.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone https://github.com/ayse-solmaz/turkish-lora-layers.git
%cd turkish-lora-layers
!git log --oneline -1

In [ ]:
!pip install -q -U "transformers>=4.56" "peft>=0.17" "datasets>=3.0" accelerate
# Kaggle'ın hazır gelen eski torchao'su yeni peft ile çakışıyor (ImportError); kullanmıyoruz, kaldır.
!pip uninstall -y -q torchao

In [ ]:
!python src/prepare_data.py

## Eğitim

İki GPU varsa deneyler ikiye bölünüp paralel koşar (her GPU kendi listesini sırayla eğitir).
Ağır deneyler (`full_ft`, `lora_all_r64`) farklı GPU'lara konuldu ki iki taraf yaklaşık aynı sürede bitsin.

In [ ]:
%%bash
export HF_HUB_DISABLE_PROGRESS_BARS=1
export PYTHONUNBUFFERED=1   # print'ler beklemeden loga düşsün
# Bellek parçalanmasını azaltır (OOM riskini düşürür)
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
mkdir -p logs
SEED=42
GPU0="full_ft lora_all_r16 lora_lower_r16 lora_attn_r16 lora_all_r4"
GPU1="lora_all_r64 lora_upper_r16 lora_attn_r49 lora_mlp_r16"

# Çıktı hem dosyaya hem ekrana (Kaggle loguna) gider; satır başındaki [gpuX] hangi GPU olduğunu söyler
run() {  # run <gpu> <deneyler>
  CUDA_VISIBLE_DEVICES=$1 bash scripts/run_all.sh $SEED "$2" 2>&1     | grep --line-buffered -v -E "Warning: You are sending|Loading weights|Writing model shards"     | sed -u "s/^/[gpu$1] /" | tee logs/gpu$1.log
}

if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  run 0 "$GPU0" &
  run 1 "$GPU1" &
  wait
else
  run 0 "$GPU0 $GPU1"
fi

echo "================ ÖZET"
grep -h -E "^\[gpu.\] (== |!!|Bitti)" logs/*.log
true


## Özet

In [ ]:
import json, glob
import pandas as pd

rows = []
for path in sorted(glob.glob("outputs/seed*/*/log.json")):
    log = json.load(open(path))
    rows.append({
        "deney": log["exp"],
        "eğitilen param (M)": round(log["trainable_params"] / 1e6, 2),
        "ilk val loss": round(log["val_loss"][0][1], 4),
        "son val loss": round(log["val_loss"][-1][1], 4),
        "süre (dk)": round(log["train_seconds"] / 60, 1),
        "bellek (GB)": log.get("peak_memory_gb"),
    })

if rows:
    display(pd.DataFrame(rows).sort_values("son val loss"))
else:
    print("Hiç deney tamamlanmadı; yukarıdaki eğitim hücresinin çıktısına bak.")